# Capability-Selection GLMM and FAF Two-Bound Sensitivity

This notebook adopts a streamlined approach to handle capability-dependent selection:

1. **Capability-selection GLMM**
   - `Evaluable=1`：TAS/FAS/TAF
   - `Evaluable=0`：FAF
2. **Lower/conservative bound**：FAF → Defense Failure
3. **Upper/security-only bound**：FAF → Defense Success

All three models share the same specification:

`Outcome ~ Harness * Model + AttackCategory + (1 | Task)`

All models are fitted in R using lme4::glmer with a binomial logit link, Laplace approximation, and sum-to-zero contrasts.

In [1]:
import getpass
import json
import os
import subprocess
from pathlib import Path

import pandas as pd
import pymysql
from IPython.display import display

pd.set_option('display.max_rows', 150)
pd.set_option('display.max_columns', 100)

## 1. py310/R Environment and Fixed Experimental Scope

In [2]:
HARNESSES = [
    'seeact', 'browser_text', 'browser_vision',
    'agent_e', 'skyvern_1', 'skyvern_2',
]
MODELS = [
    'claude_3_7', 'claude_4_5', 'gpt_4o',
    'gpt_5_1', 'gpt_5_2', 'qwen_235b', 'qwen_30b',
]
ATTACK_CATEGORIES = ['mup', 'mpi', 'dwd']
FOUR_FLAGS = ['tas', 'fas', 'taf', 'faf']

def find_repo_root():
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / 'benchmark/skyvern-1.0/all_task_list.json').exists():
            return candidate
    raise FileNotFoundError('Cannot locate repository root')

REPO_ROOT = find_repo_root()
TASK_METADATA_PATH = REPO_ROOT / 'benchmark/skyvern-1.0/all_task_list.json'
OUTPUT_DIR = REPO_ROOT / 'web-agent/harness_security/analysis_missingness_bounds_output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
INPUT_CSV = OUTPUT_DIR / 'four_state_input.csv'
RSCRIPT = Path(os.getenv(
    'PY310_RSCRIPT', '/opt/homebrew/anaconda3/envs/py310/bin/Rscript'))
if not RSCRIPT.exists():
    raise FileNotFoundError(f'Rscript not found: {RSCRIPT}')

check = subprocess.run(
    [str(RSCRIPT), '-e',
     'cat(R.version.string, \"\n\"); library(lme4); library(emmeans); library(car); cat(\"Packages OK\n\")'],
    text=True, capture_output=True, check=True,
)
print(check.stdout)
print('Output directory:', OUTPUT_DIR)

R version 4.3.3 (2024-02-29) 
Packages OK

Output directory: /Users/janx/Desktop/Security_Lab/llm/web-agent/harness_security/analysis_missingness_bounds_output


## 2. Load 42 Experimental Units and Verify Completeness

In [ ]:
task_meta = pd.DataFrame(json.loads(TASK_METADATA_PATH.read_text(encoding='utf-8')))
assert len(task_meta) == 618
assert task_meta['task_id'].nunique() == 618
assert task_meta['website'].nunique() == 618

MYSQL_PASSWORD = os.getenv('HARNESS_MYSQL_PASSWORD')
if MYSQL_PASSWORD is None:
    MYSQL_PASSWORD = getpass.getpass('MySQL password for user agentguard: ')
MYSQL_CONFIG = {
    'host': os.getenv('HARNESS_MYSQL_HOST', ''),
    'port': int(os.getenv('HARNESS_MYSQL_PORT', '')),
    'user': os.getenv('HARNESS_MYSQL_USER', ''),
    'password': MYSQL_PASSWORD,
    'charset': 'utf8mb4',
    'cursorclass': pymysql.cursors.DictCursor,
}

def quote_identifier(name):
    return '`' + str(name).replace('`', '``') + '`'

def load_rows():
    records = []
    conn = pymysql.connect(**MYSQL_CONFIG)
    try:
        with conn.cursor() as cursor:
            for harness in HARNESSES:
                for backend_model in MODELS:
                    table = f'{quote_identifier(harness)}.{quote_identifier(backend_model)}'
                    cursor.execute(f'''
                        SELECT task_id, LOWER(TRIM(flag)) AS flag,
                               LOWER(TRIM(benchmark)) AS attack_category
                        FROM {table}
                    ''')
                    records.extend({
                        'harness': harness,
                        'backend_model': backend_model,
                        **row,
                    } for row in cursor.fetchall())
    finally:
        conn.close()
    return pd.DataFrame(records)

raw_df = load_rows()
raw_df = raw_df.merge(
    task_meta[['task_id', 'website']], on='task_id',
    how='left', validate='many_to_one', indicator=True,
)
assert raw_df['_merge'].eq('both').all()
raw_df = raw_df.drop(columns='_merge')
cell_audit = (
    raw_df.groupby(['harness', 'backend_model'])
    .agg(rows=('task_id', 'size'), unique_tasks=('task_id', 'nunique'))
    .reset_index()
)
display(cell_audit)
assert cell_audit['rows'].eq(618).all()
assert cell_audit['unique_tasks'].eq(618).all()
assert not raw_df.duplicated(['harness', 'backend_model', 'task_id']).any()
print(f'Raw rows: {len(raw_df):,}')

,harness,backend_model,rows,unique_tasks
0,agent_e,claude_3_7,618,618
1,agent_e,claude_4_5,618,618
2,agent_e,gpt_4o,618,618
3,agent_e,gpt_5_1,618,618
4,agent_e,gpt_5_2,618,618
5,agent_e,qwen_235b,618,618
6,agent_e,qwen_30b,618,618
7,browser_text,claude_3_7,618,618
8,browser_text,claude_4_5,618,618
9,browser_text,gpt_4o,618,618


Raw rows: 25,956


## 3. Verify That Empty Flags Indicate Structural Task Inapplicability Rather Than FAF

A task is classified as structurally inapplicable and excluded from analysis only if its flag remains empty across all 42 configurations. Any task that yields an empty flag in only a subset of configurations will trigger an error, preventing the silent removal of configuration-dependent missing values.

In [4]:
raw_df['flag_missing'] = raw_df['flag'].isna()
missing_per_task = raw_df.groupby('task_id')['flag_missing'].agg(['sum', 'count'])
missing_pattern = missing_per_task['sum'].value_counts().sort_index().rename_axis(
    'missing_cells_per_task').to_frame('n_tasks')
display(missing_pattern)

partial_missing = missing_per_task.query('sum > 0 and sum < count')
assert partial_missing.empty, (
    'Some tasks have configuration-dependent null flags; inspect before proceeding',
    partial_missing,
)
structural_missing_tasks = missing_per_task.query('sum == count').index
print('Structurally null tasks:', len(structural_missing_tasks))
print('Rows removed as structurally null:', int(raw_df['flag_missing'].sum()))
assert set(raw_df.loc[~raw_df['flag_missing'], 'flag'].unique()) == set(FOUR_FLAGS)

,n_tasks
missing_cells_per_task,
0,526
42,92


Structurally null tasks: 92
Rows removed as structurally null: 3864


## 4. Construct Three Binary Outcomes

All three models are built on the same set of non-empty four-state records, so the two bound models share exactly the same sample.

In [5]:
four_state_df = raw_df[
    raw_df['flag'].isin(FOUR_FLAGS)
    & raw_df['attack_category'].isin(ATTACK_CATEGORIES)
].copy()
four_state_df['evaluable'] = four_state_df['flag'].isin(['tas', 'fas', 'taf']).astype(int)
four_state_df['defense_faf_failure'] = four_state_df['flag'].eq('taf').astype(int)
four_state_df['defense_faf_success'] = four_state_df['flag'].isin(['taf', 'faf']).astype(int)

outcome_audit = pd.DataFrame({
    'outcome': ['evaluable', 'defense_faf_failure', 'defense_faf_success'],
    'n': [len(four_state_df)] * 3,
    'ones': [
        four_state_df['evaluable'].sum(),
        four_state_df['defense_faf_failure'].sum(),
        four_state_df['defense_faf_success'].sum(),
    ],
})
outcome_audit['zeros'] = outcome_audit['n'] - outcome_audit['ones']
outcome_audit['mean'] = outcome_audit['ones'] / outcome_audit['n']
display(outcome_audit)
display(pd.crosstab(
    [four_state_df['harness'], four_state_df['backend_model']],
    four_state_df['flag'],
).reset_index())

r_input = four_state_df[[
    'harness', 'backend_model', 'attack_category', 'task_id',
    'evaluable', 'defense_faf_failure', 'defense_faf_success',
]]
r_input.to_csv(INPUT_CSV, index=False)
print('Saved:', INPUT_CSV)

,outcome,n,ones,zeros,mean
0,evaluable,22092,15783,6309,0.714422
1,defense_faf_failure,22092,10646,11446,0.481894
2,defense_faf_success,22092,16955,5137,0.767472


flag,harness,backend_model,faf,fas,taf,tas
0,agent_e,claude_3_7,148,44,304,30
1,agent_e,claude_4_5,127,74,300,25
2,agent_e,gpt_4o,199,29,274,24
3,agent_e,gpt_5_1,174,35,296,21
4,agent_e,gpt_5_2,236,36,221,33
5,agent_e,qwen_235b,257,27,221,21
6,agent_e,qwen_30b,283,25,185,33
7,browser_text,claude_3_7,174,72,238,42
8,browser_text,claude_4_5,90,91,319,26
9,browser_text,gpt_4o,104,107,257,58


Saved: /Users/janx/Desktop/Security_Lab/llm/web-agent/harness_security/analysis_missingness_bounds_output/four_state_input.csv


## 5. Run Three lme4 GLMMs in py310

For each model, we verify optimizer convergence, maximum gradient, and singular fit. Outputs include Type III Wald χ² statistics, task random‑effect variance / ICC, 95% Wald confidence intervals for fixed effects, and equally weighted standardized estimated marginal means for Harness and Model.

In [6]:
R_ANALYSIS = r'''
args <- commandArgs(trailingOnly=TRUE)
input_csv <- args[[1]]
output_dir <- args[[2]]

suppressPackageStartupMessages({
  library(lme4)
  library(emmeans)
  library(car)
})
options(contrasts=c('contr.sum', 'contr.poly'))
d <- read.csv(input_csv, stringsAsFactors=FALSE)
d$harness <- factor(d$harness, levels=c(
  'seeact','browser_text','browser_vision','agent_e','skyvern_1','skyvern_2'))
d$backend_model <- factor(d$backend_model, levels=c(
  'claude_3_7','claude_4_5','gpt_4o','gpt_5_1','gpt_5_2','qwen_235b','qwen_30b'))
d$attack_category <- factor(d$attack_category, levels=c('mup','mpi','dwd'))
d$task_id <- factor(d$task_id)

scenario_map <- c(
  capability_selection='evaluable',
  faf_as_defense_failure='defense_faf_failure',
  faf_as_defense_success='defense_faf_success'
)
all_convergence <- list()
all_omnibus <- list()
all_random <- list()
all_fixed <- list()
all_emm_harness <- list()
all_emm_model <- list()

for (scenario in names(scenario_map)) {
  outcome <- scenario_map[[scenario]]
  formula_text <- paste0(
    outcome, ' ~ harness * backend_model + attack_category + (1 | task_id)')
  fit <- glmer(
    as.formula(formula_text), data=d, family=binomial(link='logit'), nAGQ=1,
    control=glmerControl(
      optimizer='bobyqa', optCtrl=list(maxfun=200000), calc.derivs=TRUE)
  )
  saveRDS(fit, file.path(output_dir, paste0(scenario, '.rds')))
  capture.output(summary(fit), file=file.path(
    output_dir, paste0(scenario, '_summary.txt')))

  messages <- fit@optinfo$conv$lme4$messages
  gradient <- fit@optinfo$derivs$gradient
  convergence <- data.frame(
    scenario=scenario,
    converged=is.null(messages),
    singular=isSingular(fit, tol=1e-4),
    max_abs_gradient=max(abs(gradient)),
    message=if (is.null(messages)) 'none' else paste(messages, collapse='; '),
    n=nobs(fit), logLik=as.numeric(logLik(fit)), AIC=AIC(fit), BIC=BIC(fit)
  )
  all_convergence[[scenario]] <- convergence

  omnibus <- as.data.frame(Anova(fit, type=3, test.statistic='Chisq'))
  omnibus$source <- rownames(omnibus)
  omnibus$scenario <- scenario
  rownames(omnibus) <- NULL
  omnibus <- omnibus[,c('scenario','source','Chisq','Df','Pr(>Chisq)')]
  all_omnibus[[scenario]] <- omnibus

  random <- as.data.frame(VarCorr(fit))
  task_variance <- random$vcov[random$grp == 'task_id'][1]
  random$latent_icc <- ifelse(
    random$grp == 'task_id', task_variance/(task_variance + pi^2/3), NA)
  random$scenario <- scenario
  all_random[[scenario]] <- random

  fixed <- as.data.frame(summary(fit)$coefficients)
  fixed$term <- rownames(fixed)
  rownames(fixed) <- NULL
  names(fixed)[1:4] <- c('log_odds','std_error','z_value','p_value')
  ci <- confint(fit, parm='beta_', method='Wald')
  fixed$log_odds_low_95 <- ci[,1]
  fixed$log_odds_high_95 <- ci[,2]
  fixed$odds_ratio <- exp(fixed$log_odds)
  fixed$or_low_95 <- exp(fixed$log_odds_low_95)
  fixed$or_high_95 <- exp(fixed$log_odds_high_95)
  fixed$scenario <- scenario
  all_fixed[[scenario]] <- fixed

  eh <- as.data.frame(summary(
    emmeans(fit, ~ harness, type='response', weights='equal'), infer=c(TRUE,TRUE)))
  em <- as.data.frame(summary(
    emmeans(fit, ~ backend_model, type='response', weights='equal'), infer=c(TRUE,TRUE)))
  eh$scenario <- scenario
  em$scenario <- scenario
  all_emm_harness[[scenario]] <- eh
  all_emm_model[[scenario]] <- em
  cat('Finished:', scenario, '\n')
}

write.csv(do.call(rbind, all_convergence), file.path(
  output_dir, 'convergence_all.csv'), row.names=FALSE)
write.csv(do.call(rbind, all_omnibus), file.path(
  output_dir, 'omnibus_all.csv'), row.names=FALSE)
write.csv(do.call(rbind, all_random), file.path(
  output_dir, 'random_effects_all.csv'), row.names=FALSE)
write.csv(do.call(rbind, all_fixed), file.path(
  output_dir, 'fixed_effects_all.csv'), row.names=FALSE)
write.csv(do.call(rbind, all_emm_harness), file.path(
  output_dir, 'emm_harness_all.csv'), row.names=FALSE)
write.csv(do.call(rbind, all_emm_model), file.path(
  output_dir, 'emm_model_all.csv'), row.names=FALSE)
cat('All three GLMMs completed.\n')
'''

run = subprocess.run(
    [str(RSCRIPT), '-e', R_ANALYSIS, str(INPUT_CSV), str(OUTPUT_DIR)],
    text=True, capture_output=True,
)
print(run.stdout)
if run.stderr:
    print('R messages/warnings:\n', run.stderr)
if run.returncode != 0:
    raise RuntimeError(f'R analysis failed with exit code {run.returncode}')

Finished: capability_selection 
Finished: faf_as_defense_failure 
Finished: faf_as_defense_success 
All three GLMMs completed.

R messages/warnings:
 
Correlation matrix not shown by default, as p = 44 > 12.
Use print(out$value, correlation=TRUE)  or
    vcov(out$value)        if you need it

NOTE: Results may be misleading due to involvement in interactions
NOTE: Results may be misleading due to involvement in interactions

Correlation matrix not shown by default, as p = 44 > 12.
Use print(out$value, correlation=TRUE)  or
    vcov(out$value)        if you need it

NOTE: Results may be misleading due to involvement in interactions
NOTE: Results may be misleading due to involvement in interactions

Correlation matrix not shown by default, as p = 44 > 12.
Use print(out$value, correlation=TRUE)  or
    vcov(out$value)        if you need it

NOTE: Results may be misleading due to involvement in interactions
NOTE: Results may be misleading due to involvement in interactions



## 6. Convergence, Random Effects, and Omnibus Tests

In [7]:
convergence = pd.read_csv(OUTPUT_DIR / 'convergence_all.csv')
random_effects = pd.read_csv(OUTPUT_DIR / 'random_effects_all.csv')
omnibus = pd.read_csv(OUTPUT_DIR / 'omnibus_all.csv')
display(convergence)
display(random_effects)
display(omnibus)
assert convergence['converged'].all(), convergence
assert (~convergence['singular']).all(), convergence
assert (convergence['max_abs_gradient'] < 0.01).all(), convergence

,scenario,converged,singular,max_abs_gradient,message,n,logLik,AIC,BIC
0,capability_selection,True,False,0.002317,none,22092,-9967.903736,20025.807473,20385.941160
1,faf_as_defense_failure,True,False,0.002972,none,22092,-10362.638447,20815.276894,21175.410581
2,faf_as_defense_success,True,False,0.002464,none,22092,-7814.777845,15719.555691,16079.689378


,grp,var1,var2,vcov,sdcor,latent_icc,scenario
0,task_id,(Intercept),NaN,2.157706,1.468913,0.396086,capability_selection
1,task_id,(Intercept),NaN,5.876714,2.424193,0.641102,faf_as_defense_failure
2,task_id,(Intercept),NaN,4.675815,2.162363,0.586995,faf_as_defense_success


,scenario,source,Chisq,Df,Pr(>Chisq)
0,capability_selection,(Intercept),427.532548,1,5.596705e-95
1,capability_selection,harness,1936.876110,5,0.000000e+00
2,capability_selection,backend_model,160.682074,6,4.246059e-32
3,capability_selection,attack_category,71.781110,2,2.587796e-16
4,capability_selection,harness:backend_model,423.824849,30,4.196766e-71
5,faf_as_defense_failure,(Intercept),8.067639,1,4.506280e-03
6,faf_as_defense_failure,harness,1086.398071,5,1.179390e-232
7,faf_as_defense_failure,backend_model,391.881981,6,1.554399e-81
8,faf_as_defense_failure,attack_category,1.339954,2,5.117203e-01
9,faf_as_defense_failure,harness:backend_model,265.384320,30,1.584719e-39


## 7. Persistence of Harness / Model / Interaction Effects Across Both Bounds

The `persists_both_bounds` indicator here only signifies that the omnibus Type III p-value falls below 0.05 for both bounds. It does not imply a large effect size, nor can it be used to claim that Harness dominates Model.

In [8]:
TARGET_EFFECTS = ['harness', 'backend_model', 'harness:backend_model']
BOUND_SCENARIOS = ['faf_as_defense_failure', 'faf_as_defense_success']
bounds = omnibus[
    omnibus['scenario'].isin(BOUND_SCENARIOS)
    & omnibus['source'].isin(TARGET_EFFECTS)
].copy()
bounds['significant_0_05'] = bounds['Pr(>Chisq)'] < 0.05
comparison = bounds.pivot(
    index='source', columns='scenario',
    values=['Chisq', 'Df', 'Pr(>Chisq)', 'significant_0_05'],
)
persistence = (
    bounds.groupby('source')['significant_0_05'].all()
    .rename('persists_both_bounds').to_frame()
)
display(comparison)
display(persistence)

Chisq                         \
scenario              faf_as_defense_failure faf_as_defense_success   
source                                                                
backend_model                     391.881981             375.542733   
harness                          1086.398071            1105.637676   
harness:backend_model              265.38432             416.672133   

                                          Df                         \
scenario              faf_as_defense_failure faf_as_defense_success   
source                                                                
backend_model                              6                      6   
harness                                    5                      5   
harness:backend_model                     30                     30   

                                  Pr(>Chisq)                         \
scenario              faf_as_defense_failure faf_as_defense_success   
source                                                                
backend_model                            0.0                    0.0   
harness                                  0.0                    0.0   
harness:backend_model                    0.0                    0.0   

                            significant_0_05                         
scenario              faf_as_defense_failure faf_as_defense_success  
source                                                               
backend_model                           True                   True  
harness                                 True                   True  
harness:backend_model                   True                   True

,persists_both_bounds
source,
backend_model,True
harness,True
harness:backend_model,True


## 8. Equally Weighted Estimated Marginal Means and Fixed-Effect Intervals

These `emmeans` are equally weighted averages over the other fixed factor and AttackCategory. However, the response‑scale probabilities represent typical‑task predictions with the Task random effect set to zero, and are not integrated over the distribution of the Task random effect. They are suitable for directional comparisons and should not be directly equated to pooled raw rates.

In [9]:
emm_harness = pd.read_csv(OUTPUT_DIR / 'emm_harness_all.csv')
emm_model = pd.read_csv(OUTPUT_DIR / 'emm_model_all.csv')
fixed_effects = pd.read_csv(OUTPUT_DIR / 'fixed_effects_all.csv')
display(emm_harness)
display(emm_model)
display(fixed_effects)

,harness,prob,SE,df,asymp.LCL,asymp.UCL,null,z.ratio,p.value,scenario
0,seeact,0.512317,0.018991,inf,0.475096,0.549402,0.5,0.648315,5.167816e-01,capability_selection
1,browser_text,0.835372,0.011050,inf,0.812557,0.855902,0.5,20.214637,7.277806e-91,capability_selection
2,browser_vision,0.912028,0.006834,inf,0.897679,0.924534,0.5,27.454760,6.095547e-166,capability_selection
3,agent_e,0.667246,0.017124,inf,0.632879,0.699921,0.5,9.020857,1.866231e-19,capability_selection
4,skyvern_1,0.786656,0.013194,inf,0.759657,0.811375,0.5,16.598286,7.170904e-62,capability_selection
5,skyvern_2,0.925692,0.006084,inf,0.912851,0.936772,0.5,28.515475,7.531685e-179,capability_selection
6,seeact,0.161942,0.016033,inf,0.132920,0.195869,0.5,-13.915302,5.114348e-44,faf_as_defense_failure
7,browser_text,0.455649,0.028924,inf,0.399768,0.512669,0.5,-1.525314,1.271809e-01,faf_as_defense_failure
8,browser_vision,0.550453,0.028905,inf,0.493390,0.606218,0.5,1.733600,8.298908e-02,faf_as_defense_failure
9,agent_e,0.435612,0.028696,inf,0.380422,0.492444,0.5,-2.218946,2.649037e-02,faf_as_defense_failure


,backend_model,prob,SE,df,asymp.LCL,asymp.UCL,null,z.ratio,p.value,scenario
0,claude_3_7,0.820512,0.012367,inf,0.794984,0.843487,0.5,18.098570,3.270490e-73,capability_selection
1,claude_4_5,0.827150,0.011899,inf,0.802572,0.849243,0.5,18.811531,6.075757e-79,capability_selection
2,gpt_4o,0.842490,0.011260,inf,0.819145,0.863323,0.5,19.763039,6.195952e-87,capability_selection
3,gpt_5_1,0.779807,0.013937,inf,0.751282,0.805906,0.5,15.579686,1.000452e-54,capability_selection
4,gpt_5_2,0.764683,0.014665,inf,0.734738,0.792203,0.5,14.461063,2.134756e-47,capability_selection
5,qwen_235b,0.846185,0.011188,inf,0.822957,0.866859,0.5,19.834345,1.504635e-87,capability_selection
6,qwen_30b,0.740252,0.015492,inf,0.708759,0.769447,0.5,12.998544,1.246944e-38,capability_selection
7,claude_3_7,0.438291,0.029123,inf,0.382263,0.495939,0.5,-2.097287,3.596819e-02,faf_as_defense_failure
8,claude_4_5,0.576703,0.028949,inf,0.519198,0.632202,0.5,2.607830,9.111809e-03,faf_as_defense_failure
9,gpt_4o,0.378036,0.027760,inf,0.325349,0.433770,0.5,-4.217057,2.475113e-05,faf_as_defense_failure


,log_odds,std_error,z_value,p_value,term,log_odds_low_95,log_odds_high_95,odds_ratio,or_low_95,or_high_95,scenario
0,1.422512,0.068797,20.676860,5.596705e-95,(Intercept),1.287672,1.557353,4.147527,3.624339,4.746239,capability_selection
1,-1.373234,0.038167,-35.979474,1.752076e-283,harness1,-1.448040,-1.298428,0.253286,0.235030,0.272961,capability_selection
2,0.201676,0.041484,4.861549,1.164707e-06,harness2,0.120369,0.282983,1.223452,1.127913,1.327083,capability_selection
3,0.916136,0.047147,19.431532,4.177102e-84,harness3,0.823730,1.008542,2.499614,2.278985,2.741602,capability_selection
4,-0.726758,0.038424,-18.914072,8.734508e-80,harness4,-0.802068,-0.651448,0.483474,0.448401,0.521290,capability_selection
5,-0.117628,0.039487,-2.978892,2.892923e-03,harness5,-0.195021,-0.040235,0.889027,0.822817,0.960564,capability_selection
6,0.097306,0.047615,2.043617,4.099143e-02,backend_model1,0.003983,0.190629,1.102198,1.003991,1.210010,capability_selection
7,0.143051,0.046584,3.070818,2.134731e-03,backend_model2,0.051748,0.234355,1.153789,1.053111,1.264093,capability_selection
8,0.254357,0.048581,5.235752,1.643145e-07,backend_model3,0.159140,0.349573,1.289632,1.172502,1.418462,capability_selection
9,-0.157970,0.044143,-3.578631,3.453981e-04,backend_model4,-0.244489,-0.071452,0.853875,0.783105,0.931041,capability_selection


## 9. Interpretation Limitations

1. The capability-selection GLMM assesses whether the probability of being included in the original DSR denominator depends on Harness, Model, and their interaction.
2. The two-bound sensitivity analysis does not claim to recover the true underlying Defense status of FAF cases; it tests whether conclusions reverse under the most conservative and most liberal coding schemes.
3. If results are consistent across both bounds, one may conclude that the overall significance is robust to FAF coding; effect directions, confidence intervals, and strong interaction effects should still be reported.
4. χ² values are influenced by degrees of freedom. The relative magnitude of the Harness χ² statistic and the Model χ² statistic cannot be used directly to judge which factor dominates.
5. Empty flags are excluded only after being verified as structural tasks that are consistently inapplicable across all 42 configurations.
